# Experimento E — Adaptación progresiva (fine-tuning incremental)

Toma el modelo del Experimento A (entrenado solo en StyleGAN2) y le enseña los demás generadores **por etapas**,
de forma **encadenada**: cada fase parte del modelo que dejó la anterior.

| Fase | Datos de fine-tuning | Pregunta que responde |
|---|---|---|
| FT-1 | SG2 + StyleGAN3 | ¿Agregar SG3 mejora la detección de difusión? |
| FT-2 | SG2 + SG3 + SDXL | ¿Agregar SDXL mejora la detección de FLUX? |
| FT-3 | SG2 + SG3 + SDXL + FLUX | ¿El modelo final es robusto a todos? |

**Decisiones de diseño:**

- **Encadenado:** FT-2 parte de los pesos de FT-1, y FT-3 de los de FT-2. Es lo que hace que sea *aprendizaje
  incremental* y no tres reentrenamientos sueltos.
- **Reales constantes:** las mismas imágenes de FFHQ en train/val en las tres fases. No son objeto de estudio,
  son la referencia que permite que el clasificador binario tenga sentido. Al no cambiar, cualquier diferencia
  entre fases es atribuible solo al generador añadido.
- **Balanceo 1:1:** en cada fase se muestrean las sintéticas para que su total iguale al de las reales. Sin
  esto el desbalance crecería de 2:1 en FT-1 a 4:1 en FT-3 y contaminaría la comparación entre fases.
- **Test = CelebA-HQ:** las mismas imágenes de prueba de los Experimentos A–D, para que los resultados sean
  directamente comparables.
- **Tasa de aprendizaje reducida (1e-5):** un orden de magnitud por debajo del Experimento A. Limita el
  desplazamiento de los pesos y con ello el olvido catastrófico. Justificado porque en E se parte de un
  modelo que ya resuelve la tarea, no de pesos de ImageNet.
- **Evaluación completa:** tras cada fase se evalúa sobre los cuatro generadores. Las columnas de los
  generadores ya vistos muestran si el modelo conserva lo aprendido; las de los no vistos responden las
  preguntas de la tabla.

### Antes de correr, en Drive:
```
MyDrive/Tesis/
    codigo/config.py, nucleo.py
    datos/StyleGAN2_CelebA.zip, StyleGAN3_CelebA.zip, SDXL_Celeb_A.zip, Flux_Celeb_A.zip
    Resultados/modelos/   <-- los .pth del Experimento A ya están aquí
```

In [ ]:
!pip install timm --quiet
print("Listo")

In [ ]:
from google.colab import drive
import os, time, zipfile, glob, shutil

drive.mount('/content/drive')

RUTA_DATOS_DRIVE = '/content/drive/MyDrive/Tesis/datos'   # <-- AJUSTA
RUTA_DATOS_LOCAL = '/content/Particiones'

ZIP_DE_GENERADOR = {
    "StyleGAN2_CelebA": "StyleGAN2_CelebA.zip",   # <-- AJUSTA el nombre si difiere
    "StyleGAN3_CelebA": "StyleGAN3_CelebA.zip",
    "SDXL_CelebA":      "SDXL_Celeb_A.zip",
    "Flux_CelebA":      "Flux_Celeb_A.zip",
}
GENERADORES = list(ZIP_DE_GENERADOR)
os.makedirs(RUTA_DATOS_LOCAL, exist_ok=True)

for gen, zipname in ZIP_DE_GENERADOR.items():
    destino = os.path.join(RUTA_DATOS_LOCAL, gen)
    if os.path.isdir(os.path.join(destino, "train")):
        print(f"[{gen}] ya descomprimido")
        continue
    ruta_zip = os.path.join(RUTA_DATOS_DRIVE, zipname)
    if not os.path.exists(ruta_zip):
        raise FileNotFoundError(
            f"No encuentro {ruta_zip}\n"
            f"Archivos disponibles: {os.listdir(RUTA_DATOS_DRIVE)}"
        )
    print(f"[{gen}] descomprimiendo {zipname}...")
    t0 = time.time()
    tmp = os.path.join(RUTA_DATOS_LOCAL, f"_tmp_{gen}")
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(ruta_zip) as z:
        z.extractall(tmp)
    print(f"[{gen}] extraído en {time.time()-t0:.0f}s")
    cand = glob.glob(os.path.join(tmp, "**", "train"), recursive=True)
    if not cand:
        cand = glob.glob(os.path.join(tmp, "**", "test"), recursive=True)
    if not cand:
        raise FileNotFoundError(f"[{gen}] no encuentro 'train' ni 'test'. "
                                f"Contenido: {os.listdir(tmp)}")
    shutil.rmtree(destino, ignore_errors=True)
    shutil.move(os.path.dirname(cand[0]), destino)
    shutil.rmtree(tmp, ignore_errors=True)

print("\nCONTEO POR GENERADOR")
for gen in GENERADORES:
    linea = f"  {gen:18}"
    for split in ["train", "val", "test"]:
        for clase in ["fake", "real"]:
            d = os.path.join(RUTA_DATOS_LOCAL, gen, split, clase)
            n = len(os.listdir(d)) if os.path.isdir(d) else 0
            linea += f" | {split}/{clase}: {n:>5}"
    print(linea)

In [ ]:
import shutil, sys
from pathlib import Path

RUTA_CODIGO_DRIVE = Path('/content/drive/MyDrive/Tesis/codigo')   # <-- AJUSTA

for nombre in ['config.py', 'nucleo.py']:
    origen = RUTA_CODIGO_DRIVE / nombre
    if not origen.exists():
        raise FileNotFoundError(f"No encuentro {origen}. Súbelo a {RUTA_CODIGO_DRIVE}/")
    shutil.copy(origen, Path('/content') / nombre)

sys.path.insert(0, '/content')
import config
import nucleo

config.RUTA_PARTICIONES = Path(RUTA_DATOS_LOCAL)
config.RUTA_RESULTADOS  = Path('/content/drive/MyDrive/Tesis/Resultados2')   # <-- AJUSTA
config.RUTA_MODELOS     = config.RUTA_RESULTADOS / "modelos"
config.RUTA_METRICAS    = config.RUTA_RESULTADOS / "metricas"
config.NUM_WORKERS = 4
config.preparar_carpetas()

print("Motor importado.")
print(f"  aumento_robustez: {config.AUMENTO_ROBUSTEZ} "
      f"(blur p={config.PROB_BLUR}, jpeg p={config.PROB_JPEG})")
print(f"  epocas_max: {config.EPOCAS} | paciencia: {config.PACIENCIA_EARLY_STOPPING}")
print(f"  batch: {config.BATCH_SIZE} | lr del Experimento A: {config.LEARNING_RATE}")
print("  (el fine-tuning de E usa su propio lr, ver CELDA 5)")
print(f"  semillas: {config.SEMILLAS_ENTRENAMIENTO}")

In [ ]:
# Configuración del Experimento E
ARQUITECTURA = "efficientnet_b0"   
# ---------------------------------------------------------------------------
# TASA DE APRENDIZAJE DEL FINE-TUNING
# ---------------------------------------------------------------------------
# Se reduce a 1e-5, un orden de magnitud por debajo del Experimento A (1e-4).
#
# POR QUÉ: en cada corrección los pesos se desplazan proporcionalmente a la
# tasa de aprendizaje. Un valor alto adapta rápido al generador nuevo, pero
# mueve tanto los pesos que borra lo aprendido antes (olvido catastrófico). Un
# valor bajo mantiene el modelo cerca de donde estaba, conservando el
# conocimiento previo a costa de aprender más despacio lo nuevo. Es el dilema
# estabilidad-plasticidad del aprendizaje incremental.
#
# El cambio respecto al Experimento A está justificado porque los puntos de
# partida son distintos: en A se parte de pesos de ImageNet, lejanos a la
# tarea, lo que exige pasos grandes; en E se parte de un modelo que YA resuelve
# la tarea y solo se está ajustando.
LEARNING_RATE_FT = 1e-5
GENERADOR_BASE = "StyleGAN2_CelebA"  # el del Experimento A

# Las tres fases: qué generadores entran al conjunto de fine-tuning.
# Cada fase ACUMULA los anteriores (por eso las listas crecen).
FASES = {
    "FT-1": ["StyleGAN2_CelebA", "StyleGAN3_CelebA"],
    "FT-2": ["StyleGAN2_CelebA", "StyleGAN3_CelebA", "SDXL_CelebA"],
    "FT-3": ["StyleGAN2_CelebA", "StyleGAN3_CelebA", "SDXL_CelebA", "Flux_CelebA"],
}

# Tras cada fase se evalúa sobre los cuatro test (todos con reales CelebA-HQ)
GENERADORES_EVAL = ["StyleGAN2_CelebA", "StyleGAN3_CelebA",
                    "SDXL_CelebA", "Flux_CelebA"]

CORTO = {"StyleGAN2_CelebA": "StyleGAN2", "StyleGAN3_CelebA": "StyleGAN3",
         "SDXL_CelebA": "SDXL", "Flux_CelebA": "FLUX"}

print(f"Arquitectura: {ARQUITECTURA}")
for f, gens in FASES.items():
    print(f"  {f}: entrena con {[CORTO[g] for g in gens]}")
print(f"Evaluación tras cada fase: {[CORTO[g] for g in GENERADORES_EVAL]}")

In [ ]:
# Conjunto combinado y balanceado

# nucleo.crear_dataloader() lee UN generador. El Experimento E necesita mezclar
# varios, así que aquí se construye el conjunto a mano. Se reutiliza la misma
# función de transformaciones de nucleo.py, de modo que el aumento de robustez
# (blur y JPEG aleatorios en train) se aplica exactamente igual que en A-D.
import random
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# Convención de nucleo.py / ImageFolder: fake = 0, real = 1
ETIQUETA_FAKE, ETIQUETA_REAL = 0, 1

class ConjuntoCombinado(Dataset):
    """Conjunto formado por imágenes de varios generadores más las reales."""
    def __init__(self, muestras, entrenamiento):
        self.muestras = muestras
        self.transformar = nucleo._transforms(entrenamiento=entrenamiento)

    def __len__(self):
        return len(self.muestras)

    def __getitem__(self, i):
        ruta, etiqueta = self.muestras[i]
        imagen = Image.open(ruta).convert("RGB")
        return self.transformar(imagen), etiqueta

def construir_muestras(generadores, split, semilla):
    """Devuelve [(ruta, etiqueta), ...] con las sintéticas de varios
    generadores y las reales, en proporción 1:1.

    Las REALES se toman de un solo generador porque son idénticas en los
    cuatro (control experimental verificado): no se suman, no se repiten.

    Las SINTÉTICAS se muestrean para que su total iguale al de las reales.
    Sin este balanceo, el conjunto pasaría de 2:1 en FT-1 a 4:1 en FT-3 y las
    diferencias entre fases quedarían confundidas con el cambio de proporción.
    """
    base = Path(RUTA_DATOS_LOCAL)

    reales = sorted((base / generadores[0] / split / "real").glob("*.png"))
    if not reales:
        raise FileNotFoundError(f"Sin reales en {generadores[0]}/{split}/real")
    n_real = len(reales)

    # Cuántas sintéticas por generador para que el total sea n_real
    por_generador = n_real // len(generadores)
    rng = random.Random(semilla)

    sinteticas = []
    detalle = []
    for g in generadores:
        disponibles = sorted((base / g / split / "fake").glob("*.png"))
        elegidas = (rng.sample(disponibles, por_generador)
                    if por_generador < len(disponibles) else disponibles)
        sinteticas.extend(elegidas)
        detalle.append(f"{CORTO[g]}={len(elegidas)}")

    muestras = ([(p, ETIQUETA_FAKE) for p in sinteticas]
                + [(p, ETIQUETA_REAL) for p in reales])
    resumen = f"{split}: sintéticas {len(sinteticas)} ({', '.join(detalle)}) | reales {n_real}"
    return muestras, resumen


def crear_loader_combinado(generadores, split, semilla, barajar):
    muestras, resumen = construir_muestras(generadores, split, semilla)
    conjunto = ConjuntoCombinado(muestras, entrenamiento=(split == "train"))
    g = torch.Generator(); g.manual_seed(semilla)
    loader = DataLoader(conjunto, batch_size=config.BATCH_SIZE, shuffle=barajar,
                        num_workers=config.NUM_WORKERS,
                        pin_memory=(config.DISPOSITIVO.type == "cuda"), generator=g)
    return loader, resumen

# Comprobación rápida del balanceo en las tres fases
print("Composición del conjunto de entrenamiento por fase:")
for fase, gens in FASES.items():
    _, res = construir_muestras(gens, "train", 42)
    print(f"  {fase} -> {res}")

In [ ]:
# Entrenamiento encadenado
# Reutiliza nucleo.entrenar_una_epoca() y nucleo.evaluar() sin modificarlos:
# lo único propio es la orquestación (de qué modelo parte cada fase y con qué
# conjunto entrena).
import json, copy
from datetime import datetime

import timm
import torch
import torch.nn as nn


def entrenar_fase(modelo, generadores, semilla, ruta_guardado):
    """Fine-tuning de UNA fase. Devuelve el historial por época."""
    loader_train, res_train = crear_loader_combinado(generadores, "train", semilla, True)
    loader_val, res_val = crear_loader_combinado(generadores, "val", semilla, False)
    print(f"    {res_train}")
    print(f"    {res_val}")

    criterio = nn.CrossEntropyLoss()
    # lr reducido (1e-5, ver CELDA 5): mitiga el olvido catastrófico
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=LEARNING_RATE_FT,
                                    weight_decay=config.WEIGHT_DECAY)
    escalador = torch.amp.GradScaler(enabled=config.USAR_AMP)

    mejor_acc_val = 0.0
    sin_mejora = 0
    historial = []

    for epoca in range(1, config.EPOCAS + 1):
        perdida_train, acc_train = nucleo.entrenar_una_epoca(
            modelo, loader_train, optimizador, criterio, escalador)
        metricas_val = nucleo.evaluar(modelo, loader_val, criterio=criterio)
        acc_val = metricas_val["accuracy"]

        historial.append({"epoca": epoca,
                          "perdida_train": perdida_train,
                          "perdida_val": metricas_val["perdida"],
                          "acc_train": acc_train,
                          "acc_val": acc_val})
        print(f"      Época {epoca:2d} | loss {perdida_train:.4f}/{metricas_val['perdida']:.4f} "
              f"| acc {acc_train:.4f}/{acc_val:.4f}")

        if acc_val > mejor_acc_val:
            mejor_acc_val = acc_val
            sin_mejora = 0
            torch.save(modelo.state_dict(), ruta_guardado)
        else:
            sin_mejora += 1
            if sin_mejora >= config.PACIENCIA_EARLY_STOPPING:
                print(f"      Early stopping en época {epoca} (mejor val {mejor_acc_val:.4f})")
                break

    # Dejar el modelo en su mejor estado antes de pasar a la fase siguiente
    modelo.load_state_dict(torch.load(ruta_guardado))
    return historial, mejor_acc_val


def evaluar_en_todos(modelo, semilla):
    """Evalúa el modelo sobre el test de los cuatro generadores."""
    resultados = {}
    for gen in GENERADORES_EVAL:
        loader = nucleo.crear_dataloader(gen, "test", semilla, barajar=False)
        resultados[gen] = nucleo.evaluar(modelo, loader)
    return resultados

# --------------------------------------------------------------------------
resultados_por_semilla = []
RUTA_PARCIALES = config.RUTA_METRICAS / "parciales_E"
RUTA_PARCIALES.mkdir(parents=True, exist_ok=True)

for semilla in config.SEMILLAS_ENTRENAMIENTO:
    parcial = RUTA_PARCIALES / f"experimentoE_{ARQUITECTURA}_semilla{semilla}.json"
    if parcial.exists():
        with open(parcial, encoding="utf-8") as f:
            resultados_por_semilla.append(json.load(f))
        print(f"\n[semilla {semilla}] ya estaba hecha — la reutilizo")
        continue

    print("\n" + "=" * 70)
    print(f"SEMILLA {semilla}")
    print("=" * 70)

    nucleo.fijar_semilla(semilla)

    # Punto de partida: el modelo del Experimento A
    ruta_A = config.RUTA_MODELOS / f"{ARQUITECTURA}_{GENERADOR_BASE}_semilla{semilla}.pth"
    if not ruta_A.exists():
        raise FileNotFoundError(
            f"No encuentro el modelo del Experimento A: {ruta_A}\n"
            f"Disponibles: {sorted(p.name for p in config.RUTA_MODELOS.glob('*.pth'))}"
        )
    modelo = timm.create_model(ARQUITECTURA, pretrained=False,
                               num_classes=config.NUM_CLASES).to(config.DISPOSITIVO)
    modelo.load_state_dict(torch.load(ruta_A, map_location=config.DISPOSITIVO))
    print(f"  Punto de partida: {ruta_A.name}")

    # Evaluación del punto de partida (fila "Exp. A" de la matriz)
    eval_inicial = evaluar_en_todos(modelo, semilla)
    print("  [Exp. A] " + " | ".join(
        f"{CORTO[g]} acc {m['accuracy']:.4f}" for g, m in eval_inicial.items()))

    fases_registro = []
    for fase, generadores in FASES.items():
        print(f"\n  --- {fase}: fine-tuning con {[CORTO[g] for g in generadores]} ---")
        ruta_pth = config.RUTA_MODELOS / f"{ARQUITECTURA}_E_{fase}_semilla{semilla}.pth"

        # El modelo NO se reinicia: continúa desde la fase anterior (encadenado)
        historial, mejor_val = entrenar_fase(modelo, generadores, semilla, ruta_pth)
        evaluacion = evaluar_en_todos(modelo, semilla)

        print(f"  [{fase}] " + " | ".join(
            f"{CORTO[g]} acc {m['accuracy']:.4f}" for g, m in evaluacion.items()))

        fases_registro.append({
            "fase": fase,
            "generadores_entrenamiento": generadores,
            "mejor_acc_val": mejor_val,
            "historial": historial,
            "evaluacion": evaluacion,
            "ruta_modelo": str(ruta_pth),
        })

    registro = {
        "semilla": semilla,
        "evaluacion_inicial": eval_inicial,
        "fases": fases_registro,
    }
    resultados_por_semilla.append(registro)
    with open(parcial, "w", encoding="utf-8") as f:
        json.dump(registro, f, indent=2, ensure_ascii=False)
    print(f"\n  -> semilla {semilla} guardada en Drive")

# --------------------------------------------------------------------------
salida = {
    "experimento": "E_adaptacion_progresiva",
    "modelo": ARQUITECTURA,
    "modo": "encadenado",
    "generador_base": GENERADOR_BASE,
    "fases": {k: v for k, v in FASES.items()},
    "generadores_evaluacion": GENERADORES_EVAL,
    "fecha": datetime.now().isoformat(timespec="seconds"),
    "config": {
        "batch_size": config.BATCH_SIZE,
        "learning_rate_finetuning": LEARNING_RATE_FT,
        "learning_rate_experimento_A": config.LEARNING_RATE,
        "weight_decay": config.WEIGHT_DECAY,
        "epocas_max": config.EPOCAS,
        "paciencia": config.PACIENCIA_EARLY_STOPPING,
        "amp": config.USAR_AMP,
        "balanceo": "1:1 sintéticas/reales por muestreo",
    },
    "resultados_por_semilla": resultados_por_semilla,
}
ruta_json = config.RUTA_METRICAS / f"experimentoE_{ARQUITECTURA}.json"
with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump(salida, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 70)
print(f"EXPERIMENTO E COMPLETADO — {ARQUITECTURA}")
print("=" * 70)
print(f"-> {ruta_json}")

In [ ]:
# Matriz de adaptación (accuracy media de las 3 semillas)
# Filas = fase; columnas = generador evaluado.
# Diagonal y a la derecha: generadores aún no vistos (¿generaliza?).
# A la izquierda: generadores ya vistos (¿conserva lo aprendido?).
import numpy as np

filas = ["Exp. A"] + list(FASES)

def acc_media(fila, generador):
    vals = []
    for r in resultados_por_semilla:
        if fila == "Exp. A":
            vals.append(r["evaluacion_inicial"][generador]["accuracy"])
        else:
            f = next(x for x in r["fases"] if x["fase"] == fila)
            vals.append(f["evaluacion"][generador]["accuracy"])
    return np.mean(vals), np.std(vals, ddof=1) if len(vals) > 1 else 0.0

print(f"{'':<8}" + "".join(f"{CORTO[g]:>14}" for g in GENERADORES_EVAL))
print("-" * (8 + 14 * len(GENERADORES_EVAL)))
for fila in filas:
    linea = f"{fila:<8}"
    for g in GENERADORES_EVAL:
        m, s = acc_media(fila, g)
        linea += f"{m:>9.3f}±{s:.2f}"
    print(linea)

print("\nRecall de la clase sintética (detección de deepfakes):")
print(f"{'':<8}" + "".join(f"{CORTO[g]:>14}" for g in GENERADORES_EVAL))
print("-" * (8 + 14 * len(GENERADORES_EVAL)))
for fila in filas:
    linea = f"{fila:<8}"
    for g in GENERADORES_EVAL:
        vals = []
        for r in resultados_por_semilla:
            if fila == "Exp. A":
                vals.append(r["evaluacion_inicial"][g]["recall_fake"])
            else:
                f_ = next(x for x in r["fases"] if x["fase"] == fila)
                vals.append(f_["evaluacion"][g]["recall_fake"])
        linea += f"{np.mean(vals):>9.3f}±{np.std(vals, ddof=1):.2f}"
    print(linea)

---
## Cómo leer la matriz

**Columnas de generadores ya vistos** (a la izquierda de la fase actual) — si se mantienen altas, el modelo
conserva lo aprendido; si caen, hay olvido catastrófico.

**Columnas de generadores no vistos** (a la derecha) — responden las preguntas de tu tabla: ¿agregar StyleGAN3
mejoró la detección de difusión? ¿Agregar SDXL mejoró la de FLUX?

**Última fila (FT-3)** — el modelo final, entrenado con los cuatro. Si todas sus columnas son altas, has
demostrado que la adaptación progresiva produce un detector robusto a todos los generadores.

## Después

1. Baja `experimentoE_efficientnet_b0.json` 
2. Para correr las otras dos arquitecturas: cambia `ARQUITECTURA` en la CELDA 5 a `"resnet50"` o
   `"legacy_xception"` y vuelve a ejecutar las celdas 5 a 8.
3. Las figuras de E (matriz de adaptación, curvas por fase) las generamos después con los .json.